In [1]:
import torch
print(torch.__version__)

print(sys.version)
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.11.0+cu128
3.12.14 | packaged by Anaconda, Inc. | (main, Aug 27 2026, 14:37:13) [MSC v.1942 64 bit (AMD64)]
2.11.0+cu128
True
NVIDIA GeForce RTX 4050 Laptop GPU


In [2]:
# %pip install pandas
# %pip install numpy
# %pip install -U scikit-learn
# %pip install transformers
# %pip install emoji
# %pip install requests

In [37]:
import pandas as pd
import numpy as np
import emoji
import requests
import random

import json

# SKLearn import
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report # For accuracy score
from sklearn.preprocessing import MultiLabelBinarizer # Convert multiple emotion into format so model can train



In [4]:
# Transformer
import transformers
from transformers import AutoTokenizer
from transformers import XLMRobertaModel # AI Model download

# Import/install training model (XLM-RoBERTa)
model_name = "FacebookAI/xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = XLMRobertaModel.from_pretrained(model_name)

C:\Users\ASUS\.conda\envs\lily-ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 286.55it/s]
[transformers] XLMRobertaModel LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
# Testing purpose
text = 'I am really happy today!'
inputs = tokenizer(text, return_tensors='pt')
outputs = model(**inputs)
print(outputs.last_hidden_state.shape)

torch.Size([1, 8, 768])


In [6]:
df = pd.read_csv('emotion_data.csv')

In [7]:
df.sample(10)

,message,Happy,Sad,Angry,Frustrated,Bored,Excited,Jealous,Insecure,Relieved,Neutral
346,Yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1053,I think I did something really wrong today.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1092,Sometimes I immediately try to push the though...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1180,She looked happy.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1652,The music was too loud.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
788,Then perhaps the more useful question is not w...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21,LILY LILY LILY—GUESS WHAT JUST HAPPENED?!,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1494,I don't want them to worry.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
133,Because I ate that yesterday,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1261,But honestly I just want an apology.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Cleaning and data confirming

In [8]:
df.duplicated().sum()

np.int64(101)

In [9]:
df.isna().sum()

message          0
Happy         1737
Sad           1737
Angry         1737
Frustrated    1737
Bored         1737
Excited       1737
Jealous       1737
Insecure      1737
Relieved      1737
Neutral       1737
dtype: int64

In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1737 entries, 0 to 1736
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   message     1737 non-null   str    
 1   Happy       0 non-null      float64
 2   Sad         0 non-null      float64
 3   Angry       0 non-null      float64
 4   Frustrated  0 non-null      float64
 5   Bored       0 non-null      float64
 6   Excited     0 non-null      float64
 7   Jealous     0 non-null      float64
 8   Insecure    0 non-null      float64
 9   Relieved    0 non-null      float64
 10  Neutral     0 non-null      float64
dtypes: float64(10), str(1)
memory usage: 149.4 KB


In [11]:
df.shape

(1737, 11)

In [12]:
df.nunique()

message       1636
Happy            0
Sad              0
Angry            0
Frustrated       0
Bored            0
Excited          0
Jealous          0
Insecure         0
Relieved         0
Neutral          0
dtype: int64

In [13]:
df.columns

Index(['message', 'Happy', 'Sad', 'Angry', 'Frustrated', 'Bored', 'Excited',
       'Jealous', 'Insecure', 'Relieved', 'Neutral'],
      dtype='str')

In [14]:
df['message'] = df.message.str.strip()

In [15]:
line_break = df['message'].str.contains('\n', regex=False).sum()
print(f"Total message with line break: {line_break}")

Total message with line break: 8


In [16]:
emoji_count_message = df.message.apply(emoji.emoji_count).gt(0).sum()
print(f"Total message containing emoji: {emoji_count_message}")

emoji_count = df.message.apply(emoji.emoji_count).sum()
print(f"Total emoji count: {emoji_count}")

Total message containing emoji: 266
Total emoji count: 448


# Making new dataframe, text only

In [17]:
df_text = df.copy()

In [18]:
df_text['message'] = df_text.message.apply(
    lambda x: emoji.replace_emoji(x, replace='')
)

In [19]:
df_text.message.apply(emoji.emoji_count).sum()

np.int64(0)

In [20]:
df_text.columns

Index(['message', 'Happy', 'Sad', 'Angry', 'Frustrated', 'Bored', 'Excited',
       'Jealous', 'Insecure', 'Relieved', 'Neutral'],
      dtype='str')

## Insert local LLM model to label emotion for each message

In [39]:
sample = df_text.sample(n=100, random_state=42)

count = 0

for index, message in sample["message"].items():

    prompt = f"""
    You are an emotion annotation assistant.

    Analyze the emotional signals expressed or reasonably implied by the following message.

    Give each emotion an independent intensity score from 0 to 100.
    0 = no detectable emotion
    1-20 = very weak
    21-40 = mild
    41-60 = moderate
    61-80 = strong
    81-99 = very strong
    100 = extremely strong

    These are emotion intensity scores, NOT probabilities.
    The scores do not need to add up to 100.
    Multiple emotions can be present at the same time.
    Only give a nonzero score when there is reasonable evidence for that emotion.

    The emotions are:
    Happy
    Sad
    Angry
    Frustrated
    Bored
    Excited
    Jealous
    Insecure
    Relieved
    Neutral

    Return ONLY valid JSON using exactly these emotion names.

    Message:
    {message}
    """

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "qwen3:4b",
            "prompt": prompt,
            "stream": False
        }
    )

    result = response.json()["response"]

    try:
        emotion = json.loads(result)

        for emotion_name, score in emotion.items():
            df_text.loc[index, emotion_name] = score

    except json.JSONDecodeError:
        print(f"JSON error at row {index}")

    count += 1

    print(f"Processed: {count}/50")

Processed: 1/50
Processed: 2/50
Processed: 3/50
Processed: 4/50
Processed: 5/50
Processed: 6/50
Processed: 7/50
Processed: 8/50
Processed: 9/50
Processed: 10/50
Processed: 11/50
Processed: 12/50
Processed: 13/50
Processed: 14/50
Processed: 15/50
Processed: 16/50
Processed: 17/50
Processed: 18/50
Processed: 19/50
Processed: 20/50
Processed: 21/50
Processed: 22/50
Processed: 23/50
Processed: 24/50
Processed: 25/50
Processed: 26/50
Processed: 27/50
Processed: 28/50
Processed: 29/50
Processed: 30/50
Processed: 31/50
Processed: 32/50
Processed: 33/50
Processed: 34/50
Processed: 35/50
Processed: 36/50
Processed: 37/50
Processed: 38/50
Processed: 39/50
Processed: 40/50
Processed: 41/50
Processed: 42/50
Processed: 43/50
Processed: 44/50
Processed: 45/50
Processed: 46/50
JSON error at row 298
Processed: 47/50
Processed: 48/50
Processed: 49/50
Processed: 50/50
Processed: 51/50
Processed: 52/50
Processed: 53/50
Processed: 54/50
Processed: 55/50
Processed: 56/50
Processed: 57/50
Processed: 58/50
P

In [31]:
df_text.head(10)

,message,Happy,Sad,Angry,Frustrated,Bored,Excited,Jealous,Insecure,Relieved,Neutral
0,"Hi, Lily. Do you know who am I?",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0
1,You know that? I wanted to test something.\naA...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50.0
2,Hi Lily. How do you do?,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,80.0
3,Only sad things ever happened today... I watch...,0.0,85.0,0.0,0.0,0.0,0.0,55.0,0.0,0.0,0.0
4,"Ah well, I've never told her my feelings anywa...",0.0,25.0,0.0,0.0,0.0,0.0,0.0,70.0,0.0,0.0
5,I'm back after a break~\nHow do you do?,65.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,I like spiders.,35.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,Hihi. Debugging time~,60.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,Testing again. Second time~,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0
9,"Hi, there. How do you do?",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,85.0


In [32]:
emotion_columns = [
    "Happy", "Sad", "Angry", "Frustrated", "Bored",
    "Excited", "Jealous", "Insecure", "Relieved", "Neutral"
]

df_text.loc[df_text.index[:10], emotion_columns] = (
    df_text.loc[df_text.index[:10], emotion_columns].fillna(0)
)

In [40]:
df_text.to_csv("emotion_dataset_labeled_test.csv", index=False)

In [44]:
df_text.loc[sample.index, emotion_columns].isna().sum()

Happy         2
Sad           2
Angry         2
Frustrated    2
Bored         2
Excited       2
Jealous       2
Insecure      2
Relieved      2
Neutral       2
dtype: int64